# arXiv — Fast-DetectGPT Cross-Check

**Status: supporting / cross-check notebook, not a primary Run-All.**

**Run environment:** Google Colab + GPU. Mount Drive before running.

**Drive layout it expects:** `/content/drive/MyDrive/arxiv/processed/dec_2021/all_chunks_combined.csv`, `dec_2025/all_chunks_combined.csv`, plus the Reddit CSVs at `/content/drive/MyDrive/arxiv/reddit_pre_2022.csv` and `reddit_post_2022.csv`. These match `gdrive_data/` in the repo's Drive folder.

**Known issue:** one comparative cell expects 2021 Fast-DetectGPT scores while the corresponding 2021 scoring call is commented out. Re-enable that scoring call before running the comparison cell, or rely on the primary Binoculars notebook (`arxiv_research_articles_dataset.ipynb`) for the report's main numbers.


Cloning repo and Installing dependecies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/baoguangsheng/fast-detect-gpt.git
%cd /content/fast-detect-gpt
!pip install transformers==4.38.0 accelerate --quiet

In [ ]:
##### DELETE CORRUPTED FILES IF ANY ########
import shutil
import os

# Delete the corrupted gpt-j-6B cache
gptj_cache = "/content/drive/MyDrive/arxiv/hf_cache/models--EleutherAI--gpt-j-6B"
if os.path.exists(gptj_cache):
    shutil.rmtree(gptj_cache)
    print(f"Deleted: {gptj_cache}")

# Also clear any partial blobs/locks from the HF cache
for subdir in ["models--EleutherAI--gpt-j-6B"]:
    blob_path = f"/content/drive/MyDrive/arxiv/hf_cache/{subdir}"
    if os.path.exists(blob_path):
        shutil.rmtree(blob_path)
        print(f"Cleaned: {blob_path}")

print("\nDone. You can now re-run the model loading cell.")

In [ ]:
import sys
import os
import torch

# Add the repo paths
sys.path.insert(0, "/content/fast-detect-gpt")
sys.path.insert(0, "/content/fast-detect-gpt/scripts")

from local_infer import FastDetectGPT

# --- CONFIG ---
# Supported pairs (from the source code):
#   "gpt-j-6B" + "gpt-neo-2.7B"
#   "gpt-neo-2.7B" + "gpt-neo-2.7B"
#   "falcon-7b" + "falcon-7b-instruct"
#   "llama3-8b" + "llama3-8b-instruct"
#
# Start with gpt-j-6B + gpt-neo-2.7B (no license gating, ~12GB VRAM)
SAMPLING_MODEL = "gpt-j-6B"
SCORING_MODEL  = "gpt-neo-2.7B"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR      = "/content/drive/MyDrive/arxiv/hf_cache"

os.environ["HF_HOME"] = CACHE_DIR
os.makedirs(CACHE_DIR, exist_ok=True)

# Args object that FastDetectGPT expects
class Args:
    sampling_model_name = SAMPLING_MODEL
    scoring_model_name  = SCORING_MODEL
    device              = DEVICE
    cache_dir           = CACHE_DIR

args = Args()

print(f"Loading Fast-DetectGPT...")
print(f"  Sampling model: {SAMPLING_MODEL}")
print(f"  Scoring model:  {SCORING_MODEL}")
print(f"  This will download models on first run (may take several minutes)...")

detector = FastDetectGPT(args)
print("Loaded!")


# --- Wrapper that mimics the Binoculars API ---
class FastDetectWrapper:
    """Mimics the Binoculars API for drop-in compatibility with the existing pipeline."""

    THRESHOLD = 0.5  # AI probability above this → classified as AI

    def __init__(self, detector):
        self.detector = detector

    def _score_one(self, text: str):
        """Returns (probability, criterion, ntokens) or (None, None, 0) on error."""
        try:
            prob, crit, ntokens = self.detector.compute_prob(text)
            return float(prob), float(crit), int(ntokens)
        except Exception as e:
            print(f"  Error scoring text: {e}")
            return None, None, 0

    def compute_score(self, texts):
        """Returns AI probability (0-1). Accepts a string or list of strings."""
        if isinstance(texts, str):
            return self._score_one(texts)[0]
        return [self._score_one(t)[0] for t in texts]

    def compute_criterion(self, texts):
        """Returns raw Fast-DetectGPT criterion scores."""
        if isinstance(texts, str):
            return self._score_one(texts)[1]
        return [self._score_one(t)[1] for t in texts]

    def predict(self, texts):
        """Returns 'Most likely AI-Generated' or 'Most likely Human-Generated'."""
        if isinstance(texts, str):
            texts = [texts]
            single = True
        else:
            single = False

        results = []
        for t in texts:
            prob, _, _ = self._score_one(t)
            if prob is None:
                results.append("error")
            elif prob >= self.THRESHOLD:
                results.append("Most likely AI-Generated")
            else:
                results.append("Most likely Human-Generated")

        return results[0] if single else results


# Instantiate
fdg = FastDetectWrapper(detector)

#Sanity Check

In [ ]:
# --- Sanity check ---
test_human = ("Regardless of the results, I think we need to make sure that we are positioning ourselves "
              "for maximum follow-on impact. That's what they are going to be lookig for after "
              "this project. We need to show that our models are better at detecting than the next best thing regardless of what that is and that we can move quicker than the next best thing also regardless of what that thing is. What are you thoughts?")

test_ai = ("In an increasingly interconnected world, the pace at which information flows has fundamentally reshaped how individuals and organizations make decisions. What once required prolonged deliberation and access to limited data can now be accomplished in moments, often with a surplus of insights drawn from diverse sources. However, this abundance introduces its own challenges. The ability to discern signal from noise has become a critical skill, demanding not only technical proficiency but also thoughtful judgment and contextual awareness. As systems grow more complex, the importance of adaptability continues to rise. Individuals are expected to navigate ambiguity, synthesize incomplete information, and respond to evolving circumstances with clarity and precision."
"At the same time, the role of intentionality cannot be overstated. Progress is no longer defined solely by speed or scale, but by the alignment of actions with broader objectives and long-term value creation. This requires a shift from reactive patterns of behavior toward more deliberate, forward-looking strategies. Whether in professional environments or everyday life, the capacity to anticipate downstream effects and make decisions accordingly has become a distinguishing factor. Ultimately, those who are able to balance efficiency with reflection, and innovation with purpose, are best positioned to thrive in a landscape defined by constant change and expanding possibility.")

print(f"\n--- Sanity check ---")
print(f"Human-like text:")
print(f"  AI probability: {fdg.compute_score(test_human):.4f}")
print(f"  Prediction: {fdg.predict(test_human)}")

print(f"\nAI-like text:")
print(f"  AI probability: {fdg.compute_score(test_ai):.4f}")
print(f"  Prediction: {fdg.predict(test_ai)}")

# Detection Pipeline End-to-End

In [ ]:
# @title
# =============================================================================
# CELL 1: Configuration
# =============================================================================
import os
import time
import json
import pandas as pd
import numpy as np
from datetime import datetime

# --- CONFIG ---
CSV_2021 = "/content/drive/MyDrive/arxiv/processed/dec_2021/all_chunks_combined.csv"  # <-- CHANGE
CSV_2025 = "/content/drive/MyDrive/arxiv/processed/dec_2025/all_chunks_combined.csv"  # <-- CHANGE
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/fast_detect_results"
BATCH_SIZE = 1  # Fast-DetectGPT processes one text at a time internally

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# CELL 2: Scoring function (uses fdg from previous cells)
# =============================================================================
def score_dataset(csv_path, output_dir, fdg_model, dataset_label):
    """Score all chunks in a CSV with Fast-DetectGPT."""

    df = pd.read_csv(csv_path)
    total = len(df)
    print(f"\nScoring {dataset_label}: {total} chunks")

    # Resume from checkpoint if exists
    checkpoint_path = os.path.join(output_dir, f"checkpoint_{dataset_label}.json")
    start_idx = 0
    probs = [None] * total
    crits = [None] * total
    preds = [None] * total

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            ck = json.load(f)
        start_idx = ck["next_idx"]
        probs = ck["probs"]
        crits = ck["crits"]
        preds = ck["preds"]
        print(f"  Resuming from {start_idx}/{total}")

    times = []
    for i in range(start_idx, total):
        t_start = time.time()
        text = str(df["text"].iloc[i]) if pd.notna(df["text"].iloc[i]) else ""

        if len(text.strip()) == 0:
            probs[i] = None
            crits[i] = None
            preds[i] = "skipped_empty"
        else:
            try:
                prob, crit, _ = fdg_model.detector.compute_prob(text)
                probs[i] = float(prob)
                crits[i] = float(crit)
                preds[i] = "Most likely AI-Generated" if prob >= 0.5 else "Most likely Human-Generated"
            except Exception as e:
                probs[i] = None
                crits[i] = None
                preds[i] = f"error: {str(e)[:50]}"

        times.append(time.time() - t_start)

        # Save checkpoint every 50 chunks
        if (i + 1) % 50 == 0 or i == total - 1:
            with open(checkpoint_path, "w") as f:
                json.dump({
                    "next_idx": i + 1, "probs": probs,
                    "crits": crits, "preds": preds,
                }, f)

            avg_time = np.mean(times[-50:])
            remaining = (total - i - 1) * avg_time / 60
            print(f"  [{i+1}/{total}] {1/avg_time:.1f} chunks/s | ETA: {remaining:.1f} min")

    # Save final results
    df["fastdetect_probability"] = probs
    df["fastdetect_criterion"] = crits
    df["fastdetect_prediction"] = preds

    output_csv = os.path.join(output_dir, f"scored_{dataset_label}.csv")
    df.to_csv(output_csv, index=False)
    print(f"  Saved to: {output_csv}")

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    # Summary
    valid = [p for p in probs if p is not None]
    if valid:
        ai_count = sum(1 for p in valid if p >= 0.5)
        print(f"\n  {dataset_label} Summary:")
        print(f"    Mean AI probability: {np.mean(valid):.4f}")
        print(f"    Median AI probability: {np.median(valid):.4f}")
        print(f"    Classified as AI: {ai_count}/{len(valid)} ({ai_count/len(valid)*100:.1f}%)")

    return df


# =============================================================================
# CELL 3: Run scoring on both datasets
# =============================================================================
# print("=" * 70)
# print("SCORING: December 2021 (Pre-LLM)")
# print("=" * 70)
# df_2021_scored = score_dataset(CSV_2021, OUTPUT_DIR, fdg, "dec_2021")

print("\n" + "=" * 70)
print("SCORING: December 2025 (Post-LLM)")
print("=" * 70)
df_2025_scored = score_dataset(CSV_2025, OUTPUT_DIR, fdg, "dec_2025")


# =============================================================================
# CELL 4: Comparative analysis
# =============================================================================
import matplotlib.pyplot as plt

print("\n" + "=" * 70)
print("COMPARATIVE SUMMARY — Fast-DetectGPT")
print("=" * 70)

p21 = df_2021_scored["fastdetect_probability"].dropna()
p25 = df_2025_scored["fastdetect_probability"].dropna()

print(f"\n{'Metric':<35} {'Dec 2021':>12} {'Dec 2025':>12} {'Delta':>12}")
print("-" * 75)
print(f"{'Mean AI probability':<35} {p21.mean():>12.4f} {p25.mean():>12.4f} {p25.mean()-p21.mean():>+12.4f}")
print(f"{'Median AI probability':<35} {p21.median():>12.4f} {p25.median():>12.4f} {p25.median()-p21.median():>+12.4f}")

# Multi-threshold analysis
print(f"\nMulti-Threshold Detection Rates:")
print(f"{'Threshold':<12} {'2021 AI %':>12} {'2025 AI %':>12} {'Difference':>12}")
print("-" * 50)
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    r21 = (p21 >= t).mean() * 100
    r25 = (p25 >= t).mean() * 100
    print(f"{t:<12.2f} {r21:>11.1f}% {r25:>11.1f}% {r25-r21:>+11.1f}pp")

# Statistical test
from scipy import stats
u, p_val = stats.mannwhitneyu(p21, p25, alternative="two-sided")
cohens_d = (p25.mean() - p21.mean()) / np.sqrt((p21.std()**2 + p25.std()**2) / 2)
print(f"\nMann-Whitney U test: U={u:.0f}, p={p_val:.2e}")
print(f"Cohen's d: {cohens_d:.3f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Fast-DetectGPT Score Distributions", fontsize=15, fontweight="bold")

C_2021 = "#2E86AB"
C_2025 = "#E84855"

ax = axes[0]
ax.hist(p21, bins=50, alpha=0.6, color=C_2021, label="Dec 2021", density=True, edgecolor="white", linewidth=0.3)
ax.hist(p25, bins=50, alpha=0.6, color=C_2025, label="Dec 2025", density=True, edgecolor="white", linewidth=0.3)
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, label="Threshold (0.5)")
ax.set_xlabel("AI Probability")
ax.set_ylabel("Density")
ax.set_title("AI Probability Distribution")
ax.legend(frameon=False)

ax = axes[1]
labels = ["Dec 2021\n(Pre-LLM)", "Dec 2025\n(Post-LLM)"]
ai_rates = [(p21 >= 0.5).mean() * 100, (p25 >= 0.5).mean() * 100]
ax.bar(range(len(labels)), ai_rates, color=[C_2021, C_2025], edgecolor="white", width=0.5)
for i, rate in enumerate(ai_rates):
    ax.text(i, rate + 0.5, f"{rate:.1f}%", ha="center", fontweight="bold", fontsize=12)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("% Classified as AI-Generated")
ax.set_title("AI Detection Rate")

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "fastdetect_results.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\nFigure saved to: {fig_path}")

#Data Analysis from CSV

In [ ]:
# =============================================================================
# CELL 1: Load data and configure
# =============================================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# --- CONFIG ---
CSV_2021 = "/content/drive/MyDrive/arxiv/fast_detect_results/scored_dec_2021.csv"  # <-- CHANGE
CSV_2025 = "/content/drive/MyDrive/arxiv/fast_detect_results/scored_dec_2025.csv"  # <-- CHANGE
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/fast_detect_results/figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- PLOT STYLE ---
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

C_2021 = "#2E86AB"
C_2025 = "#E84855"

# Load data
df_2021 = pd.read_csv(CSV_2021)
df_2025 = pd.read_csv(CSV_2025)
df_2021["period"] = "Dec 2021"
df_2025["period"] = "Dec 2025"

# Drop any failed scores
df_2021 = df_2021.dropna(subset=["fastdetect_probability"])
df_2025 = df_2025.dropna(subset=["fastdetect_probability"])

print(f"Loaded {len(df_2021)} chunks from 2021, {len(df_2025)} from 2025")
print(f"Unique papers — 2021: {df_2021['paper_id'].nunique()}, 2025: {df_2025['paper_id'].nunique()}")


# =============================================================================
# CELL 2: Statistical summary
# =============================================================================
print("\n" + "=" * 70)
print("STATISTICAL SUMMARY")
print("=" * 70)

p21 = df_2021["fastdetect_probability"]
p25 = df_2025["fastdetect_probability"]
c21 = df_2021["fastdetect_criterion"]
c25 = df_2025["fastdetect_criterion"]

print(f"\n{'Metric':<35} {'Dec 2021':>14} {'Dec 2025':>14} {'Delta':>14}")
print("-" * 80)
for name, s21, s25 in [
    ("Mean AI probability",   p21.mean(),   p25.mean()),
    ("Median AI probability", p21.median(), p25.median()),
    ("Std AI probability",    p21.std(),    p25.std()),
    ("Mean criterion",        c21.mean(),   c25.mean()),
    ("Median criterion",      c21.median(), c25.median()),
]:
    print(f"{name:<35} {s21:>14.4f} {s25:>14.4f} {s25-s21:>+14.4f}")

# Multi-threshold detection rates
print(f"\n\nMulti-Threshold AI Detection Rates:")
print(f"{'Threshold':<12} {'2021 AI %':>12} {'2025 AI %':>12} {'Difference':>14}")
print("-" * 55)
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
for t in thresholds:
    r21 = (p21 >= t).mean() * 100
    r25 = (p25 >= t).mean() * 100
    print(f"{t:<12.2f} {r21:>11.1f}% {r25:>11.1f}% {r25-r21:>+12.1f}pp")

# Statistical tests
u, p_val = stats.mannwhitneyu(p21, p25, alternative="two-sided")
cohens_d = (p25.mean() - p21.mean()) / np.sqrt((p21.std()**2 + p25.std()**2) / 2)
ks_stat, ks_p = stats.ks_2samp(p21, p25)

print(f"\n\nStatistical Tests:")
print(f"  Mann-Whitney U:  U={u:.0f}, p={p_val:.2e}")
print(f"  Kolmogorov-Smirnov: D={ks_stat:.4f}, p={ks_p:.2e}")
print(f"  Cohen's d:       {cohens_d:.4f}")


# =============================================================================
# CELL 3: Main visualization (4-panel figure)
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Fast-DetectGPT: arXiv Dec 2021 vs Dec 2025",
             fontsize=16, fontweight="bold", y=1.00)

# Panel 1: Probability distribution
ax = axes[0][0]
bins = np.linspace(0, 1, 50)
ax.hist(p21, bins=bins, alpha=0.6, color=C_2021, label="Dec 2021 (Pre-LLM)",
        density=True, edgecolor="white", linewidth=0.3)
ax.hist(p25, bins=bins, alpha=0.6, color=C_2025, label="Dec 2025 (Post-LLM)",
        density=True, edgecolor="white", linewidth=0.3)
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.7, label="Threshold (0.5)")
ax.set_xlabel("AI Probability")
ax.set_ylabel("Density")
ax.set_title("AI Probability Distribution", fontweight="bold")
ax.legend(frameon=False)

# Panel 2: Cumulative distribution (clearest way to see distribution shifts)
ax = axes[0][1]
sorted_21 = np.sort(p21)
sorted_25 = np.sort(p25)
ax.plot(sorted_21, np.arange(len(sorted_21)) / len(sorted_21),
        color=C_2021, linewidth=2, label="Dec 2021")
ax.plot(sorted_25, np.arange(len(sorted_25)) / len(sorted_25),
        color=C_2025, linewidth=2, label="Dec 2025")
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("AI Probability")
ax.set_ylabel("Cumulative Proportion")
ax.set_title("Cumulative Distribution Function", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.2)

# Panel 3: Detection rate at multiple thresholds
ax = axes[1][0]
threshold_range = np.linspace(0.1, 0.95, 30)
rates_21 = [(p21 >= t).mean() * 100 for t in threshold_range]
rates_25 = [(p25 >= t).mean() * 100 for t in threshold_range]
ax.plot(threshold_range, rates_21, color=C_2021, linewidth=2, marker='o', markersize=4, label="Dec 2021")
ax.plot(threshold_range, rates_25, color=C_2025, linewidth=2, marker='s', markersize=4, label="Dec 2025")
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5, label="Default (0.5)")
ax.set_xlabel("Probability Threshold")
ax.set_ylabel("% Classified as AI-Generated")
ax.set_title("Detection Rate vs Threshold", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.2)

# Panel 4: Detection rate bar at default threshold
ax = axes[1][1]
labels = ["Dec 2021\n(Pre-LLM)", "Dec 2025\n(Post-LLM)"]
ai_rates = [(p21 >= 0.5).mean() * 100, (p25 >= 0.5).mean() * 100]
bars = ax.bar(range(len(labels)), ai_rates, color=[C_2021, C_2025],
              edgecolor="white", width=0.5)
for i, (rate, bar) in enumerate(zip(ai_rates, bars)):
    ax.text(i, rate + max(ai_rates) * 0.02, f"{rate:.1f}%",
            ha="center", fontweight="bold", fontsize=13)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("% Classified as AI-Generated")
ax.set_title("AI Detection Rate (threshold=0.5)", fontweight="bold")
ax.set_ylim(0, max(ai_rates) * 1.25 if max(ai_rates) > 0 else 5)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "01_fastdetect_main_results.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\nSaved: {fig_path}")


# =============================================================================
# CELL 4: Per-paper aggregation
# =============================================================================
print("\n" + "=" * 70)
print("PER-PAPER ANALYSIS")
print("=" * 70)

paper_21 = df_2021.groupby("paper_id")["fastdetect_probability"].agg(["mean", "median", "max", "count"])
paper_25 = df_2025.groupby("paper_id")["fastdetect_probability"].agg(["mean", "median", "max", "count"])

# Papers with majority of chunks classified as AI
maj_ai_21 = df_2021.groupby("paper_id").apply(lambda g: (g["fastdetect_probability"] >= 0.5).mean() > 0.5).sum()
maj_ai_25 = df_2025.groupby("paper_id").apply(lambda g: (g["fastdetect_probability"] >= 0.5).mean() > 0.5).sum()

print(f"\n{'Metric':<40} {'Dec 2021':>12} {'Dec 2025':>12}")
print("-" * 70)
print(f"{'Total papers':<40} {len(paper_21):>12} {len(paper_25):>12}")
print(f"{'Mean of paper-mean probabilities':<40} {paper_21['mean'].mean():>12.4f} {paper_25['mean'].mean():>12.4f}")
print(f"{'Papers with majority AI chunks':<40} {maj_ai_21:>11} ({maj_ai_21/len(paper_21)*100:.1f}%) {maj_ai_25:>11} ({maj_ai_25/len(paper_25)*100:.1f}%)")
print(f"{'Papers with any AI chunk (prob>=0.5)':<40} {(paper_21['max']>=0.5).sum():>11} ({(paper_21['max']>=0.5).mean()*100:.1f}%) {(paper_25['max']>=0.5).sum():>11} ({(paper_25['max']>=0.5).mean()*100:.1f}%)")

# Per-paper distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Per-Paper Mean AI Probability", fontsize=15, fontweight="bold")

ax = axes[0]
ax.hist(paper_21["mean"], bins=40, alpha=0.6, color=C_2021, label="Dec 2021",
        density=True, edgecolor="white", linewidth=0.3)
ax.hist(paper_25["mean"], bins=40, alpha=0.6, color=C_2025, label="Dec 2025",
        density=True, edgecolor="white", linewidth=0.3)
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Mean AI Probability per Paper")
ax.set_ylabel("Density")
ax.set_title("Distribution of Paper-Level Scores")
ax.legend(frameon=False)

ax = axes[1]
ax.boxplot([paper_21["mean"], paper_25["mean"]],
           labels=["Dec 2021", "Dec 2025"],
           patch_artist=True, widths=0.5,
           boxprops=dict(facecolor=C_2021, alpha=0.6),
           medianprops=dict(color="black", linewidth=2))
# Color the second box differently
ax.findobj(plt.matplotlib.patches.PathPatch)
for patch, color in zip(ax.patches, [C_2021, C_2025]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel("Mean AI Probability per Paper")
ax.set_title("Paper-Level Score Comparison")
ax.grid(alpha=0.2, axis='y')

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "02_per_paper_analysis.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\nSaved: {fig_path}")


# =============================================================================
# CELL 5: Score by zone (position in paper)
# =============================================================================
if "zone" in df_2021.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle("AI Probability by Position in Paper", fontsize=15, fontweight="bold")

    zone_labels = {0: "Intro", 1: "Early-Mid", 2: "Late-Mid", 3: "Conclusion"}
    zones = sorted(df_2021["zone"].unique())

    for df_s, label, color in [(df_2021, "Dec 2021", C_2021), (df_2025, "Dec 2025", C_2025)]:
        means = df_s.groupby("zone")["fastdetect_probability"].mean()
        sems = df_s.groupby("zone")["fastdetect_probability"].sem()
        x_pos = [zone_labels.get(z, f"Zone {z}") for z in zones]
        ax.errorbar(x_pos, [means[z] for z in zones], yerr=[sems[z] for z in zones],
                    color=color, marker='o', markersize=8, linewidth=2,
                    capsize=5, label=label)

    ax.axhline(y=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5, label="Threshold")
    ax.set_ylabel("Mean AI Probability")
    ax.set_title("Writing Style by Position")
    ax.legend(frameon=False)
    ax.grid(alpha=0.2)

    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, "03_zone_analysis.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {fig_path}")


# =============================================================================
# CELL 6: Summary text output
# =============================================================================
print("\n" + "=" * 70)
print("KEY FINDINGS — Fast-DetectGPT")
print("=" * 70)

ai_21 = (p21 >= 0.5).mean() * 100
ai_25 = (p25 >= 0.5).mean() * 100

print(f"\n2021 (Pre-LLM): {ai_21:.1f}% of chunks classified as AI-generated")
print(f"2025 (Post-LLM): {ai_25:.1f}% of chunks classified as AI-generated")
print(f"Difference: {ai_25 - ai_21:+.1f}pp")
print(f"\nMean probability shift: {p25.mean() - p21.mean():+.4f}")
print(f"Effect size (Cohen's d): {cohens_d:.4f}")

if abs(cohens_d) < 0.1:
    interp = "negligible"
elif abs(cohens_d) < 0.3:
    interp = "small"
elif abs(cohens_d) < 0.5:
    interp = "moderate"
else:
    interp = "large"
print(f"Effect size interpretation: {interp}")

if p_val < 0.001:
    print(f"Statistical significance: highly significant (p < 0.001)")
elif p_val < 0.05:
    print(f"Statistical significance: significant (p = {p_val:.3f})")
else:
    print(f"Statistical significance: not significant (p = {p_val:.3f})")

#Testing Positive Control Data

In [ ]:
# =============================================================================
# CELL 1: Load rewrites and re-score with Fast-DetectGPT
# =============================================================================
import os
import time
import pandas as pd
import numpy as np

# --- CONFIG ---
# Path to your existing rewrites CSV from the GPT-4o positive control
REWRITES_CSV = "/content/drive/MyDrive/arxiv/positive_control_gpt4o/rewritten_chunks.csv"  # <-- CHANGE
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/fast_detect_results/positive_control"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the rewrites (assumes fdg is already loaded from earlier cells)
df = pd.read_csv(REWRITES_CSV)
df = df.dropna(subset=["rewritten_text"])
print(f"Loaded {len(df)} rewrites from {df['rewrite_mode'].nunique()} modes")
print(f"Modes: {df['rewrite_mode'].unique().tolist()}")


# =============================================================================
# CELL 2: Score originals AND rewrites with Fast-DetectGPT
# =============================================================================
# We need to score both because the existing CSV has Binoculars scores for
# originals, not Fast-DetectGPT scores

print("\nScoring originals with Fast-DetectGPT...")
orig_probs = []
orig_crits = []

# Get unique originals (each appears once per rewrite mode)
unique_originals = df.drop_duplicates(subset=["paper_id", "chunk_index"])[
    ["paper_id", "chunk_index", "original_text"]
].reset_index(drop=True)

orig_results = {}
start = time.time()
for i, row in unique_originals.iterrows():
    text = str(row["original_text"])
    try:
        prob, crit, _ = fdg.detector.compute_prob(text)
        orig_results[(row["paper_id"], row["chunk_index"])] = (float(prob), float(crit))
    except Exception as e:
        orig_results[(row["paper_id"], row["chunk_index"])] = (None, None)

    if (i + 1) % 25 == 0:
        elapsed = time.time() - start
        eta = (len(unique_originals) - i - 1) * elapsed / (i + 1) / 60
        print(f"  [{i+1}/{len(unique_originals)}] ETA: {eta:.1f} min")

print(f"\nScoring rewrites with Fast-DetectGPT...")
rewrite_probs = []
rewrite_crits = []
start = time.time()
for i, row in df.iterrows():
    text = str(row["rewritten_text"])
    try:
        prob, crit, _ = fdg.detector.compute_prob(text)
        rewrite_probs.append(float(prob))
        rewrite_crits.append(float(crit))
    except Exception as e:
        rewrite_probs.append(None)
        rewrite_crits.append(None)

    if (i + 1) % 50 == 0:
        elapsed = time.time() - start
        eta = (len(df) - i - 1) * elapsed / (i + 1) / 60
        print(f"  [{i+1}/{len(df)}] ETA: {eta:.1f} min")

# Add scores to dataframe
df["fastdetect_original_prob"] = [orig_results.get((r["paper_id"], r["chunk_index"]), (None, None))[0] for _, r in df.iterrows()]
df["fastdetect_original_crit"] = [orig_results.get((r["paper_id"], r["chunk_index"]), (None, None))[1] for _, r in df.iterrows()]
df["fastdetect_rewrite_prob"] = rewrite_probs
df["fastdetect_rewrite_crit"] = rewrite_crits

# Save
output_csv = os.path.join(OUTPUT_DIR, "positive_control_fastdetect.csv")
df.to_csv(output_csv, index=False)
print(f"\nSaved: {output_csv}")


# =============================================================================
# CELL 3: Analysis and visualization
# =============================================================================
import matplotlib.pyplot as plt
from scipy import stats

# Filter valid scores
df_valid = df.dropna(subset=["fastdetect_original_prob", "fastdetect_rewrite_prob"])
print(f"\nValid pairs for analysis: {len(df_valid)}")

modes = df_valid["rewrite_mode"].unique()

print("\n" + "=" * 80)
print("POSITIVE CONTROL — Fast-DetectGPT")
print("=" * 80)
print(f"\n{'Mode':<15} {'Orig Prob':>12} {'Rewrite Prob':>14} {'Delta':>10} {'Orig AI%':>10} {'Rewrite AI%':>12}")
print("-" * 80)

for mode in modes:
    sub = df_valid[df_valid["rewrite_mode"] == mode]
    o_mean = sub["fastdetect_original_prob"].mean()
    r_mean = sub["fastdetect_rewrite_prob"].mean()
    o_ai = (sub["fastdetect_original_prob"] >= 0.5).mean() * 100
    r_ai = (sub["fastdetect_rewrite_prob"] >= 0.5).mean() * 100
    print(f"{mode:<15} {o_mean:>12.4f} {r_mean:>14.4f} {r_mean-o_mean:>+10.4f} {o_ai:>9.1f}% {r_ai:>11.1f}%")

# Overall
o_mean = df_valid["fastdetect_original_prob"].mean()
r_mean = df_valid["fastdetect_rewrite_prob"].mean()
o_ai = (df_valid["fastdetect_original_prob"] >= 0.5).mean() * 100
r_ai = (df_valid["fastdetect_rewrite_prob"] >= 0.5).mean() * 100
print(f"\n{'OVERALL':<15} {o_mean:>12.4f} {r_mean:>14.4f} {r_mean-o_mean:>+10.4f} {o_ai:>9.1f}% {r_ai:>11.1f}%")

# Statistical tests
print("\n\nPaired Statistical Tests:")
for mode in modes:
    sub = df_valid[df_valid["rewrite_mode"] == mode]
    o = sub["fastdetect_original_prob"]
    r = sub["fastdetect_rewrite_prob"]
    t_stat, p_val = stats.ttest_rel(o, r)
    cohens_d = (r.mean() - o.mean()) / np.sqrt((o.std()**2 + r.std()**2) / 2)
    print(f"  {mode:<15} t={t_stat:>7.3f}, p={p_val:.2e}, Cohen's d={cohens_d:+.3f}")

# --- Visualization ---
C_ORIG = "#2E86AB"
C_REWRITE = "#E84855"
mode_colors = ["#E84855", "#FF9F1C", "#7B2D8E"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Positive Control: Fast-DetectGPT on Original vs LLM-Rewritten Text",
             fontsize=15, fontweight="bold", y=1.01)

# Panel 1: Score distributions
ax = axes[0][0]
bins = np.linspace(0, 1, 40)
ax.hist(df_valid["fastdetect_original_prob"], bins=bins, alpha=0.6, color=C_ORIG,
        label="Original (human)", density=True, edgecolor="white", linewidth=0.3)
for i, mode in enumerate(modes):
    sub = df_valid[df_valid["rewrite_mode"] == mode]
    ax.hist(sub["fastdetect_rewrite_prob"], bins=bins, alpha=0.4, color=mode_colors[i % len(mode_colors)],
            label=f"Rewrite ({mode})", density=True, edgecolor="white", linewidth=0.3)
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlabel("AI Probability")
ax.set_ylabel("Density")
ax.set_title("Score Distributions")
ax.legend(frameon=False, fontsize=9)

# Panel 2: Paired scatter
ax = axes[0][1]
for i, mode in enumerate(modes):
    sub = df_valid[df_valid["rewrite_mode"] == mode]
    ax.scatter(sub["fastdetect_original_prob"], sub["fastdetect_rewrite_prob"],
               alpha=0.4, s=20, color=mode_colors[i % len(mode_colors)], label=mode)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, linewidth=1, label="y=x (no change)")
ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.5)
ax.axvline(x=0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Original AI Probability")
ax.set_ylabel("Rewritten AI Probability")
ax.set_title("Paired Comparison")
ax.legend(frameon=False, fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Panel 3: Score shift box plot
ax = axes[1][0]
shift_data = []
for mode in modes:
    sub = df_valid[df_valid["rewrite_mode"] == mode]
    shift = (sub["fastdetect_rewrite_prob"] - sub["fastdetect_original_prob"]).values
    shift_data.append(shift)
bp = ax.boxplot(shift_data, labels=[m.replace("_", "\n") for m in modes],
                patch_artist=True, widths=0.6)
for patch, color in zip(bp["boxes"], mode_colors[:len(modes)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.axhline(y=0, color="black", linestyle="--", linewidth=1)
ax.set_ylabel("Probability Shift (rewrite − original)")
ax.set_title("Score Change by Rewrite Mode")
ax.grid(alpha=0.2, axis='y')

# Panel 4: Detection rates
ax = axes[1][1]
mode_labels = [m.replace("_", "\n") for m in modes]
orig_rates = [(df_valid[df_valid["rewrite_mode"] == m]["fastdetect_original_prob"] >= 0.5).mean() * 100 for m in modes]
rewrite_rates = [(df_valid[df_valid["rewrite_mode"] == m]["fastdetect_rewrite_prob"] >= 0.5).mean() * 100 for m in modes]

x = np.arange(len(modes))
width = 0.35
ax.bar(x - width/2, orig_rates, width, color=C_ORIG, label="Original", edgecolor="white")
ax.bar(x + width/2, rewrite_rates, width, color=C_REWRITE, label="Rewritten", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(mode_labels)
ax.set_ylabel("% Classified as AI")
ax.set_title("AI Detection Rate (threshold=0.5)")
ax.legend(frameon=False)
for i, (o, r) in enumerate(zip(orig_rates, rewrite_rates)):
    ax.text(i - width/2, o + max(max(orig_rates), max(rewrite_rates)) * 0.02, f"{o:.0f}%", ha="center", fontsize=9)
    ax.text(i + width/2, r + max(max(orig_rates), max(rewrite_rates)) * 0.02, f"{r:.0f}%", ha="center", fontsize=9)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "positive_control_fastdetect.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\nSaved: {fig_path}")

# --- Key takeaway ---
print("\n" + "=" * 70)
print("KEY FINDING")
print("=" * 70)
print(f"\nOriginal text (human) flagged as AI: {o_ai:.1f}%")
print(f"LLM-rewritten text flagged as AI:    {r_ai:.1f}%")
print(f"Difference: {r_ai - o_ai:+.1f}pp")

if r_ai - o_ai > 30:
    print("\n→ Fast-DetectGPT effectively distinguishes human from LLM-rewritten academic text.")
    print("  This is a much stronger signal than Binoculars showed.")
elif r_ai - o_ai > 10:
    print("\n→ Fast-DetectGPT shows moderate discrimination on academic text.")
    print("  Better than Binoculars but still imperfect.")
else:
    print("\n→ Fast-DetectGPT also struggles with academic text.")
    print("  Both detectors fail in this domain — supporting the broader limitation hypothesis.")

#Reddit Dataset Deteciton

In [ ]:
# =============================================================================
# CELL 1: Configuration
# =============================================================================
import os
import time
import json
import pandas as pd
import numpy as np

CSV_PRE = "/content/drive/MyDrive/arxiv/reddit_pre_2022.csv"    # <-- CHANGE
CSV_POST = "/content/drive/MyDrive/arxiv/reddit_post_2022.csv"  # <-- CHANGE
OUTPUT_DIR = "/content/drive/MyDrive/arxiv/fast_detect_results/reddit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MIN_WORDS = 50
MAX_WORDS = 1000


# =============================================================================
# CELL 2: Scoring function with checkpointing
# =============================================================================
def score_reddit_dataset(csv_path, output_dir, fdg_model, label):
    df = pd.read_csv(csv_path)
    print(f"\n[{label}] Loaded {len(df)} posts")

    df = df[(df["word_count"] >= MIN_WORDS) & (df["word_count"] <= MAX_WORDS)].reset_index(drop=True)
    print(f"[{label}] After word count filter: {len(df)} posts")

    checkpoint_path = os.path.join(output_dir, f"checkpoint_{label}.json")
    start_idx = 0
    probs = [None] * len(df)
    crits = [None] * len(df)

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            ck = json.load(f)
        start_idx = ck["next_idx"]
        probs = ck["probs"]
        crits = ck["crits"]
        while len(probs) < len(df):
            probs.append(None)
            crits.append(None)
        print(f"[{label}] Resuming from {start_idx}/{len(df)}")

    print(f"[{label}] Scoring {len(df) - start_idx} posts...")
    times = []

    for i in range(start_idx, len(df)):
        t_start = time.time()
        text = str(df["text"].iloc[i]) if pd.notna(df["text"].iloc[i]) else ""

        if len(text.strip()) == 0 or len(text.split()) < 10:
            probs[i] = None
            crits[i] = None
        else:
            try:
                prob, crit, _ = fdg_model.detector.compute_prob(text)
                probs[i] = float(prob)
                crits[i] = float(crit)
            except Exception:
                probs[i] = None
                crits[i] = None

        times.append(time.time() - t_start)

        if (i + 1) % 100 == 0 or i == len(df) - 1:
            with open(checkpoint_path, "w") as f:
                json.dump({"next_idx": i + 1, "probs": probs, "crits": crits}, f)
            avg_time = np.mean(times[-100:])
            remaining = (len(df) - i - 1) * avg_time / 60
            print(f"  [{i+1}/{len(df)}] {1/avg_time:.1f} posts/s | ETA: {remaining:.1f} min")

    df["fastdetect_probability"] = probs
    df["fastdetect_criterion"] = crits

    output_csv = os.path.join(output_dir, f"reddit_scored_{label}.csv")
    df.to_csv(output_csv, index=False)
    print(f"[{label}] Saved to: {output_csv}")

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    return df


# =============================================================================
# CELL 3: Score both datasets (assumes fdg is loaded)
# =============================================================================
print("=" * 70)
print("SCORING PRE-2022 REDDIT")
print("=" * 70)
df_pre = score_reddit_dataset(CSV_PRE, OUTPUT_DIR, fdg, "pre_2022")

print("\n" + "=" * 70)
print("SCORING POST-2022 REDDIT")
print("=" * 70)
df_post = score_reddit_dataset(CSV_POST, OUTPUT_DIR, fdg, "post_2022")


# =============================================================================
# CELL 4: Comparative analysis
# =============================================================================
import matplotlib.pyplot as plt
from scipy import stats

df_pre = df_pre.dropna(subset=["fastdetect_probability"])
df_post = df_post.dropna(subset=["fastdetect_probability"])

p_pre = df_pre["fastdetect_probability"]
p_post = df_post["fastdetect_probability"]

print("\n" + "=" * 70)
print("COMPARATIVE SUMMARY — Reddit Pre-2022 vs Post-2022")
print("=" * 70)

print(f"\n{'Metric':<35} {'Pre-2022':>14} {'Post-2022':>14} {'Delta':>14}")
print("-" * 80)
print(f"{'N posts':<35} {len(p_pre):>14} {len(p_post):>14}")
print(f"{'Mean AI probability':<35} {p_pre.mean():>14.4f} {p_post.mean():>14.4f} {p_post.mean()-p_pre.mean():>+14.4f}")
print(f"{'Median AI probability':<35} {p_pre.median():>14.4f} {p_post.median():>14.4f} {p_post.median()-p_pre.median():>+14.4f}")
print(f"{'Std AI probability':<35} {p_pre.std():>14.4f} {p_post.std():>14.4f}")

print(f"\nMulti-Threshold Detection Rates:")
print(f"{'Threshold':<12} {'Pre-2022 AI%':>14} {'Post-2022 AI%':>15} {'Difference':>14}")
print("-" * 60)
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    r_pre = (p_pre >= t).mean() * 100
    r_post = (p_post >= t).mean() * 100
    print(f"{t:<12.2f} {r_pre:>13.1f}% {r_post:>14.1f}% {r_post-r_pre:>+13.1f}pp")

u, p_val = stats.mannwhitneyu(p_pre, p_post, alternative="two-sided")
cohens_d = (p_post.mean() - p_pre.mean()) / np.sqrt((p_pre.std()**2 + p_post.std()**2) / 2)
ks_stat, ks_p = stats.ks_2samp(p_pre, p_post)

print(f"\nStatistical Tests:")
print(f"  Mann-Whitney U: U={u:.0f}, p={p_val:.2e}")
print(f"  Kolmogorov-Smirnov: D={ks_stat:.4f}, p={ks_p:.2e}")
print(f"  Cohen's d: {cohens_d:.4f}")

# By year (post only, since pre is one period)
print(f"\n\nPost-2022 Detection Rate by Year:")
for year in sorted(df_post["year"].unique()):
    sub = df_post[df_post["year"] == year]
    rate = (sub["fastdetect_probability"] >= 0.5).mean() * 100
    print(f"  {year}: {len(sub)} posts, {rate:.1f}% AI, mean prob {sub['fastdetect_probability'].mean():.4f}")


# =============================================================================
# CELL 5: Visualization
# =============================================================================
C_PRE = "#2E86AB"
C_POST = "#E84855"

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Fast-DetectGPT: Reddit Pre-2022 vs Post-2022",
             fontsize=17, fontweight="bold", y=1.00)

# Panel 1: Probability distribution
ax = axes[0][0]
bins = np.linspace(0, 1, 50)
ax.hist(p_pre, bins=bins, alpha=0.6, color=C_PRE, label="Pre-2022",
        density=True, edgecolor="white", linewidth=0.3)
ax.hist(p_post, bins=bins, alpha=0.6, color=C_POST, label="Post-2022",
        density=True, edgecolor="white", linewidth=0.3)
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlabel("AI Probability")
ax.set_ylabel("Density")
ax.set_title("AI Probability Distribution", fontweight="bold")
ax.legend(frameon=False)

# Panel 2: CDF
ax = axes[0][1]
ax.plot(np.sort(p_pre), np.arange(len(p_pre))/len(p_pre),
        color=C_PRE, linewidth=2, label="Pre-2022")
ax.plot(np.sort(p_post), np.arange(len(p_post))/len(p_post),
        color=C_POST, linewidth=2, label="Post-2022")
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("AI Probability")
ax.set_ylabel("Cumulative Proportion")
ax.set_title("Cumulative Distribution", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.2)

# Panel 3: Detection rate vs threshold
ax = axes[0][2]
thresh_range = np.linspace(0.1, 0.95, 30)
ax.plot(thresh_range, [(p_pre >= t).mean()*100 for t in thresh_range],
        color=C_PRE, linewidth=2, marker='o', markersize=4, label="Pre-2022")
ax.plot(thresh_range, [(p_post >= t).mean()*100 for t in thresh_range],
        color=C_POST, linewidth=2, marker='s', markersize=4, label="Post-2022")
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Threshold")
ax.set_ylabel("% Classified as AI")
ax.set_title("Detection Rate vs Threshold", fontweight="bold")
ax.legend(frameon=False)
ax.grid(alpha=0.2)

# Panel 4: By year (combining pre + post)
ax = axes[1][0]
df_all = pd.concat([df_pre, df_post])
year_stats = df_all.groupby("year").agg(
    n=("fastdetect_probability", "count"),
    ai_rate=("fastdetect_probability", lambda x: (x >= 0.5).mean() * 100),
    mean_prob=("fastdetect_probability", "mean"),
).sort_index()
colors = [C_PRE if y < 2022 else C_POST for y in year_stats.index]
ax.bar(year_stats.index.astype(str), year_stats["ai_rate"],
       color=colors, edgecolor="white")
for i, (y, v) in enumerate(zip(year_stats.index, year_stats["ai_rate"])):
    ax.text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=9)
ax.set_xlabel("Year")
ax.set_ylabel("% Classified as AI")
ax.set_title("AI Detection Rate by Year", fontweight="bold")
ax.axvline(x=0.5 if 2021 in year_stats.index else -1,
           color="gray", linestyle=":", alpha=0.5)

# Panel 5: By subreddit (top 10 by combined volume, showing pre vs post)
ax = axes[1][1]
top_subs = df_all["subreddit"].value_counts().head(10).index.tolist()
sub_pre_rates = []
sub_post_rates = []
for sub in top_subs:
    pre_sub = df_pre[df_pre["subreddit"] == sub]["fastdetect_probability"]
    post_sub = df_post[df_post["subreddit"] == sub]["fastdetect_probability"]
    sub_pre_rates.append((pre_sub >= 0.5).mean() * 100 if len(pre_sub) > 0 else 0)
    sub_post_rates.append((post_sub >= 0.5).mean() * 100 if len(post_sub) > 0 else 0)

y_pos = np.arange(len(top_subs))
width = 0.4
ax.barh(y_pos - width/2, sub_pre_rates, width, color=C_PRE, label="Pre-2022", edgecolor="white")
ax.barh(y_pos + width/2, sub_post_rates, width, color=C_POST, label="Post-2022", edgecolor="white")
ax.set_yticks(y_pos)
ax.set_yticklabels(top_subs, fontsize=9)
ax.set_xlabel("% Classified as AI")
ax.set_title("Top Subreddits: Pre vs Post", fontweight="bold")
ax.invert_yaxis()
ax.legend(frameon=False)

# Panel 6: Overall detection rates
ax = axes[1][2]
labels = ["Pre-2022", "Post-2022"]
rates = [(p_pre >= 0.5).mean() * 100, (p_post >= 0.5).mean() * 100]
bars = ax.bar(labels, rates, color=[C_PRE, C_POST], edgecolor="white", width=0.5)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, rate + max(rates)*0.02,
            f"{rate:.1f}%", ha="center", fontweight="bold", fontsize=13)
ax.set_ylabel("% Classified as AI")
ax.set_title("Overall AI Detection Rate", fontweight="bold")
ax.set_ylim(0, max(rates) * 1.25 if max(rates) > 0 else 5)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "reddit_comparison.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\nSaved: {fig_path}")


# =============================================================================
# CELL 6: Key findings
# =============================================================================
print("\n" + "=" * 70)
print("KEY FINDINGS — Reddit Domain")
print("=" * 70)

ai_pre = (p_pre >= 0.5).mean() * 100
ai_post = (p_post >= 0.5).mean() * 100

print(f"\nPre-2022 Reddit AI detection rate:  {ai_pre:.1f}%")
print(f"Post-2022 Reddit AI detection rate: {ai_post:.1f}%")
print(f"Difference: {ai_post - ai_pre:+.1f}pp")
print(f"\nMean probability shift: {p_post.mean() - p_pre.mean():+.4f}")
print(f"Effect size (Cohen's d): {cohens_d:.4f}")

if abs(cohens_d) < 0.1:
    interp = "negligible"
elif abs(cohens_d) < 0.3:
    interp = "small"
elif abs(cohens_d) < 0.5:
    interp = "moderate"
else:
    interp = "large"
print(f"Effect size interpretation: {interp}")

print("\n--- Comparison to arXiv results ---")
print("This lets you compare domains: Reddit (informal writing) vs arXiv (academic)")
print("If Reddit shows a larger pre/post difference, it suggests either:")
print("  (a) LLM adoption is more prevalent in Reddit than academic writing, or")
print("  (b) Detectors work better on informal text where human/AI patterns diverge more")